# 09 — RAG corpus: inventory, download and text extraction

**Purpose.** Build the list of official documents the RAG may use, download them once, extract clean text, and record everything with hashes and licences. Contract and open questions: see notebook 08.

- **Inputs:** `configs/rag.yaml`, live pages and APIs at `stat.fi`, `tyomarkkinatori.fi`, `doria.fi` and `julkaisut.valtioneuvosto.fi`.
- **Outputs:**
  - `data/raw/rag/<doc_id>.pdf|html`: original files (git-ignored; most are restricted-use)
  - `data/raw/rag/text/<doc_id>.txt`: extracted text (git-ignored)
  - `data/manifests/rag_corpus_<timestamp>.json`: inventory, hashes, licences (committed)

**Safe by default.** `DRY_RUN = True` lists what would be downloaded and its size, and downloads nothing. Set it to `False` to fetch.

**Sources**

| ID | Source | Period | Language |
|---|---|---|---|
| A | Statistics Finland job vacancy releases | 3 newest releases | en |
| B | KEHA Employment Bulletins (DORIA) | Jan 2025 – now | en, fi |
| B2 | Ministry (TEM) Employment Bulletins, government publication archive | 2013 – early 2025 | fi |

Anything older than these pages list is **not** collected; see "Known gaps" at the end.

In [ ]:
# Mount Drive on Colab; skipped automatically when running locally.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
    os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"
except ImportError:
    pass

In [ ]:
# Only needed on a fresh runtime; the project requirements already list these.
# !pip install -q requests certifi trafilatura beautifulsoup4 lxml pyyaml pandas pypdf

In [ ]:
import os, re, json, time, hashlib, datetime as dt, importlib.metadata
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, unquote, quote
import requests, certifi, yaml
import pandas as pd
from bs4 import BeautifulSoup

def _find_repo():
    env = os.environ.get("JOBAI_REPO")
    if env:
        return Path(env).resolve()
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "configs" / "rag.yaml").is_file():
            return cand
    return p

REPO = _find_repo()
RAW_RAG = REPO / "data" / "raw" / "rag"
TEXT_DIR = RAW_RAG / "text"
MAN = REPO / "data" / "manifests"
for d in (RAW_RAG, TEXT_DIR, MAN):
    d.mkdir(parents=True, exist_ok=True)

CFG = yaml.safe_load(open(REPO / "configs" / "rag.yaml"))
NOW_UTC = dt.datetime.now(dt.timezone.utc)
TS = NOW_UTC.strftime("%Y%m%dT%H%M%SZ")

DRY_RUN = True          # False = actually download files
REQUEST_DELAY_S = 1.0   # be polite to public servers
TIMEOUT = 60
HEADERS = {"User-Agent": "JobAI-course-project/0.1 (research use; see repository README)"}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def _fix_redirect(response, *args, **kwargs):
    # doria.fi redirects Finnish filenames (e.g. ...hein%c3%a426.pdf) with a raw
    # latin-1 byte in the Location header, which makes `requests` crash while it
    # follows the redirect. Rewrite the header as a valid, percent-encoded URL.
    loc = response.headers.get("Location")
    if loc:
        raw = loc.encode("latin-1", errors="ignore")
        try:
            text = raw.decode("utf-8")
        except UnicodeDecodeError:
            text = raw.decode("latin-1")
        response.headers["Location"] = quote(text, safe=":/?&=%#~@+,;!$'()*-._[]")
    return response

SESSION.hooks["response"].append(_fix_redirect)

def get(url, retries=4, **kw):
    """GET with a polite delay and retries: public servers occasionally reset connections."""
    for attempt in range(retries):
        time.sleep(REQUEST_DELAY_S * (attempt + 1))   # waits longer after each failure
        try:
            r = SESSION.get(url, timeout=TIMEOUT, verify=certifi.where(), **kw)
            if r.status_code >= 500 and attempt < retries - 1:
                continue
            r.raise_for_status()
            return r
        except (requests.ConnectionError, requests.Timeout):
            if attempt == retries - 1:
                raise

print("repo   :", REPO)
print("dry run:", DRY_RUN)

In [ ]:
import unicodedata, io, zipfile

# Month names in English and Finnish (used to read the period a bulletin describes).
MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6, "july": 7,
    "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
    "tammikuu": 1, "helmikuu": 2, "maaliskuu": 3, "huhtikuu": 4, "toukokuu": 5, "kesäkuu": 6,
    "heinäkuu": 7, "elokuu": 8, "syyskuu": 9, "lokakuu": 10, "marraskuu": 11, "joulukuu": 12,
}
FI_MONTH_NAMES = ["tammikuu", "helmikuu", "maaliskuu", "huhtikuu", "toukokuu", "kesäkuu",
                  "heinäkuu", "elokuu", "syyskuu", "lokakuu", "marraskuu", "joulukuu"]

def clean_title(text):
    # Titles from these sites contain zero-width and non-breaking spaces.
    return re.sub(r"[​﻿\xa0\s]+", " ", text or "").strip()

def period_from_title(title):
    """(year, month) a bulletin describes, from titles like 'Employment Bulletin July 2026'
    or 'Työllisyyskatsaus, joulukuu 2018'; None if not parseable."""
    m = re.search(r"([A-Za-zäöåÄÖÅ]+)\s+(20\d\d)", title or "")
    if not m or m.group(1).lower() not in MONTHS:
        return None
    return int(m.group(2)), MONTHS[m.group(1).lower()]

def bulletin_month_end(title):
    """First day after the month a bulletin describes; None if not parseable."""
    period = period_from_title(title)
    if not period:
        return None
    year, month = period
    return dt.date(year + (month == 12), month % 12 + 1, 1)

## Source A — Statistics Finland job vacancy releases

The release list on `stat.fi/en/statistics/atp` is rendered in the page for the **3 most recent** releases only; the full archive is loaded by JavaScript and cannot be read this way. Each release is a short HTML page.

**Licence:** CC BY 4.0 for texts, tables and graphs (Statistics Finland terms of use, checked 2026-09-21). Attribution is required.

In [ ]:
ATP_URL = "https://stat.fi/en/statistics/atp"

STAT_LICENCE = "CC BY 4.0 (Statistics Finland terms of use)"

def discover_statfin():
    soup = BeautifulSoup(get(ATP_URL).text, "lxml")
    rows, seen = [], set()
    for a in soup.find_all("a", href=True):
        if "/publication/" not in a["href"]:
            continue
        url = urljoin(ATP_URL, a["href"])
        if url in seen:
            continue
        seen.add(url)
        # The listing shows the release date (e.g. 20/08/2026) in a parent element
        # of the link; walk up until the nearest ancestor that contains one.
        published, node = None, a
        for _ in range(4):
            node = node.find_parent()
            if node is None:
                break
            m = re.search(r"(\d{2})/(\d{2})/(\d{4})", node.get_text(" ", strip=True))
            if m:
                published = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
                break
        rows.append({
            "doc_id": "statfin_atp_" + url.rstrip("/").split("/")[-1],
            "source_id": "A", "source_type": "statfin_release_page",
            "title": clean_title(a.get_text(" ", strip=True)), "language": "en",
            "published": published, "landing_url": url, "file_url": url,
            "format": "html", "licence": STAT_LICENCE,
        })
    return rows

statfin_rows = discover_statfin()
pd.DataFrame(statfin_rows)[["doc_id", "published", "title"]]

## Source B — KEHA Employment Bulletins

Työmarkkinatori lists bulletins from February 2025 with permanent `urn.fi` links (some are wrapped in an email-safety redirect and are unwrapped below). Each URN resolves to a record on DORIA, the national repository, which gives the PDF link, publication date, language and rights.

**Licence:** DORIA records these as **In Copyright 1.0**. That is not an open licence. The files are used here for a course project and are **not committed to git**. Confirm the terms with KEHA before any public release.

The English and Finnish pages are both read, so Finnish bulletins stay in Finnish (`configs/rag.yaml`).

In [ ]:
BULLETIN_PAGES = {
    "en": "https://tyomarkkinatori.fi/en/employment-and-statistics/evaluation-and-research/employment-bulletin",
    "fi": "https://tyomarkkinatori.fi/tyollisyys-ja-tilastot/arviointi-ja-tutkimus/tyollisyyskatsaus",
}
URN_RE = re.compile(r"URN:NBN:fi-fe\d+", re.I)

def unwrap(href):
    # Email-safety wrappers put the real link in the ?url= parameter.
    if "safelinks.protection.outlook.com" in href:
        href = unquote(parse_qs(urlparse(href).query).get("url", [href])[0])
    return href

def discover_bulletin_urns():
    found = {}
    for lang, page in BULLETIN_PAGES.items():
        soup = BeautifulSoup(get(page).text, "lxml")
        for a in soup.find_all("a", href=True):
            m = URN_RE.search(unquote(unwrap(a["href"])))
            if m:
                urn = m.group(0).upper().replace("URN:NBN:FI-FE", "URN:NBN:fi-fe")
                found.setdefault(urn, {"page_language": lang, "link_text": a.get_text(" ", strip=True)})
    return found

KEHA_LICENCE = "In Copyright (not open): (c) KEHA Centre"

def normalise_rights(raw):
    # DORIA rights are inconsistent: "In Copyright 1.0" on most records, a bare
    # "KEHA Centre" / "KEHA-keskus" or nothing on others. Treat all as not open.
    return KEHA_LICENCE

def read_doria(urn, page_language=None):
    landing = get(f"https://urn.fi/{urn}").url
    soup = BeautifulSoup(get(landing).text, "lxml")
    def meta(name):
        tag = soup.find("meta", attrs={"name": name})
        return tag["content"].strip() if tag and tag.get("content") else None
    pdf = meta("citation_pdf_url")
    return {
        "doc_id": "keha_bulletin_" + urn.split("fi-fe")[-1],
        "source_id": "B", "source_type": "keha_bulletin",
        "title": clean_title(meta("citation_title") or meta("DC.title")),
        "language": meta("citation_language") or meta("DC.language") or page_language,
        "published": meta("citation_date"),
        "landing_url": landing, "file_url": pdf, "format": "pdf",
        "licence": normalise_rights(meta("DC.rights")),
        "licence_raw": meta("DC.rights"),
        "urn": urn,
    }

bulletin_urns = discover_bulletin_urns()
print(len(bulletin_urns), "bulletin URNs found")
bulletin_rows, bulletin_errors = [], []
for urn, info in bulletin_urns.items():
    try:
        bulletin_rows.append(read_doria(urn, info["page_language"]))
    except Exception as exc:
        bulletin_errors.append({"urn": urn, "error": str(exc)[:200]})
print(len(bulletin_rows), "resolved;", len(bulletin_errors), "failed")
bulletin_rows[:2]

## Source B2 — Ministry (TEM) Employment Bulletins, 2013 to early 2025

Before KEHA took over in 2025, the labour ministry published the bulletins. They live in the government publication archive, which has a search API, so they can be listed reliably instead of scraped.

| Period | What the archive holds | Licence (archive `dc.rights`) |
|---|---|---|
| 2013–2015 | One **zip per year** holding 12 monthly PDFs | cc by-sa 4.0 (open) |
| 2016 – Feb 2025 | One **PDF per month** | Copyrighted, personal use only, commercial use prohibited |

All are **Finnish**. Each zip is opened with a few small range requests (no full download) so every month becomes its own document with its own date. Zip members have no publication date in the archive, so it is **estimated as the last day of the month after the one described** (bulletins appear within about a month). That can only hide a document slightly too long, never expose a future one.

Bulletins that KEHA (source B) already provides for the same month and language are dropped as duplicates.

In [ ]:
VN_API = "https://julkaisut.valtioneuvosto.fi/server/api"
JSON_HDR = {"Accept": "application/json"}
YEARLY_ZIP_YEARS = range(2013, 2016)   # the data starts in 2013Q1; monthly PDFs start in 2016
VN_MONTHLY_LICENCE = "Copyrighted: personal use only, commercial use prohibited (archive dc.rights)"
VN_ZIP_LICENCE = "CC BY-SA 4.0 (archive dc.rights)"
ZIP_MONTH_STEMS = {"tammi": 1, "helmi": 2, "maalis": 3, "huhti": 4, "touko": 5, "kesa": 6,
                   "heina": 7, "elo": 8, "syys": 9, "loka": 10, "marras": 11, "joulu": 12}

def vn_search(query, max_pages=5):
    found = []
    for page in range(max_pages):
        r = get(f"{VN_API}/discover/search/objects", headers=JSON_HDR,
                params={"query": query, "size": 100, "page": page, "sort": "dc.date.issued,asc"})
        objs = r.json()["_embedded"]["searchResult"]["_embedded"]["objects"]
        if not objs:
            break
        found += [o["_embedded"]["indexableObject"] for o in objs]
    return found

def vn_meta(item, key):
    values = item["metadata"].get(key) or [{}]
    return values[0].get("value")

def vn_files(uuid):
    item = get(f"{VN_API}/core/items/{uuid}", headers=JSON_HDR, params={"embed": "bundles/bitstreams"}).json()
    files = []
    for bundle in item["_embedded"]["bundles"]["_embedded"]["bundles"]:
        if bundle["name"] != "ORIGINAL":
            continue
        for b in bundle["_embedded"]["bitstreams"]["_embedded"]["bitstreams"]:
            files.append({"name": b["name"], "bytes": b["sizeBytes"], "url": b["_links"]["content"]["href"]})
    return files

class HttpRangeFile(io.RawIOBase):
    """Read-only file over HTTP range requests, so a zip's file list can be read without downloading it."""
    def __init__(self, url):
        self.url, self.pos = url, 0
        head = get(url, headers={"Range": "bytes=0-0"})
        self.size = int(head.headers["Content-Range"].split("/")[-1])
    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self.pos
    def seek(self, offset, whence=0):
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos
    def read(self, n=-1):
        n = self.size - self.pos if n < 0 else n
        if n == 0 or self.pos >= self.size:
            return b""
        end = min(self.pos + n, self.size) - 1
        body = get(self.url, headers={"Range": f"bytes={self.pos}-{end}"}).content
        self.pos += len(body)
        return body
    def readinto(self, buf):
        data = self.read(len(buf))
        buf[:len(data)] = data
        return len(data)

def month_from_filename(name):
    ascii_name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode().lower()
    hits = {m for stem, m in ZIP_MONTH_STEMS.items() if stem in ascii_name}
    return hits.pop() if len(hits) == 1 else None

def last_day_of_next_month(year, month):
    # first day of month+2, minus one day (handles the year wrap)
    y, m = year + (month + 2 > 12), (month + 1) % 12 + 1
    return dt.date(y, m, 1) - dt.timedelta(days=1)

def vn_row(year, month, **fields):
    title = f"Työllisyyskatsaus, {FI_MONTH_NAMES[month - 1]} {year}"
    return {"doc_id": f"tem_bulletin_{year}_{month:02d}", "source_id": "B2",
            "source_type": "tem_bulletin", "title": title, "language": "fi", "format": "pdf", **fields}

def discover_vn_archive():
    items = {}
    for query in ("Työllisyyskatsaus", "Työllisyyskatsaukset vuodelta"):
        for item in vn_search(query):
            items[item["uuid"]] = item
    rows, notes = [], []
    for uuid, item in items.items():
        title = clean_title(vn_meta(item, "dc.title"))
        rights = vn_meta(item, "dc.rights")
        landing = vn_meta(item, "dc.identifier.uri")
        yearly = re.fullmatch(r"Työllisyyskatsaukset vuodelta (\d{4})", title)
        if title.startswith("Työllisyyskatsaus, "):
            period, pdfs = period_from_title(title), [f for f in vn_files(uuid) if f["name"].lower().endswith(".pdf")]
            if not period or not pdfs:
                notes.append(f"skipped (no period or PDF): {title}")
                continue
            rows.append(vn_row(*period, published=(vn_meta(item, "dc.date.issued") or "")[:10] or None,
                               landing_url=landing, file_url=pdfs[0]["url"], zip_member=None,
                               expected_bytes=pdfs[0]["bytes"], licence=VN_MONTHLY_LICENCE,
                               licence_raw=rights, date_source="archive record"))
        elif yearly and int(yearly.group(1)) in YEARLY_ZIP_YEARS:
            year = int(yearly.group(1))
            zips = [f for f in vn_files(uuid) if f["name"].lower().endswith(".zip")]
            if not zips:
                notes.append(f"skipped (no zip): {title}")
                continue
            members = [i for i in zipfile.ZipFile(HttpRangeFile(zips[0]["url"])).infolist()
                       if i.filename.lower().endswith(".pdf")]
            seen_months = {}
            for info in members:
                month = month_from_filename(info.filename)
                if month is None or month in seen_months:
                    notes.append(f"skipped zip member {info.filename} ({year}): month not identified")
                    continue
                seen_months[month] = info.filename
                rows.append(vn_row(year, month, published=last_day_of_next_month(year, month).isoformat(),
                                   landing_url=landing, file_url=zips[0]["url"], zip_member=info.filename,
                                   expected_bytes=info.file_size, licence=VN_ZIP_LICENCE,
                                   licence_raw=rights, date_source="estimated"))
            if len(seen_months) != 12:
                notes.append(f"{year} zip: found {len(seen_months)} of 12 months")
    return rows, notes

vn_rows, vn_notes = discover_vn_archive()
print(len(vn_rows), "archive bulletins found")
for n in vn_notes:
    print("NOTE:", n)

## Inventory

One table for both sources. Check it before downloading: dates, languages, licences, and rows with a missing PDF link.

In [ ]:
inventory = pd.DataFrame(statfin_rows + bulletin_rows + vn_rows)
inventory["published"] = inventory["published"].str[:10]

# Drop archive bulletins that KEHA (source B) already provides for the same month and language.
def period_key(row):
    return (period_from_title(row["title"]), row["language"])

keha_keys = {period_key(r) for r in inventory.to_dict("records") if r["source_id"] == "B"}
is_dup = inventory.apply(lambda r: r["source_id"] == "B2" and period_key(r) in keha_keys, axis=1)
print("duplicates dropped (already in source B):", int(is_dup.sum()))
inventory = inventory[~is_dup]

# The archive can list one month twice: in a yearly zip and again as a monthly item
# (e.g. December 2015). Keep the monthly item, which has a real publication date.
same_month = inventory["doc_id"].duplicated(keep=False)
zip_copy = same_month & (inventory["date_source"] == "estimated")
print("zip copies dropped (monthly item exists):", int(zip_copy.sum()))
inventory = inventory[~zip_copy]
assert not inventory["doc_id"].duplicated().any(), inventory[inventory["doc_id"].duplicated(keep=False)]
inventory = inventory.sort_values(["source_id", "published"], ascending=[True, False]).reset_index(drop=True)

problems = inventory[inventory["file_url"].isna() | inventory["published"].isna()]
print("documents:", len(inventory), "| with missing file_url or date:", len(problems))
display(inventory.groupby(["source_id", "language", "licence"]).agg(
    n=("doc_id", "size"), first=("published", "min"), last=("published", "max")).reset_index())
display(inventory.groupby(inventory["published"].str[:4]).size().rename("documents per year").to_frame().T)

### Date sanity check

The `published` date decides which documents retrieval may use (contract, rule 3). Some DORIA records carry a wrong year, for example a *December 2025* bulletin dated January 2025. A bulletin cannot be published before the month it describes, so those dates are flagged and `published_effective` is corrected by one year when that fits, otherwise set to the **first day after the bulletin month**. Both choices can only hide a document slightly too long, never expose a future one to an earlier forecast. Retrieval must use `published_effective`.

In [ ]:
def effective_date(row):
    published = dt.date.fromisoformat(row["published"])
    if row["source_id"] not in ("B", "B2"):
        return published, False
    earliest = bulletin_month_end(row["title"])
    if earliest and published < earliest:
        # Most bad records are one year early: try that correction first,
        # otherwise fall back to the first day after the bulletin month.
        try:
            fixed = published.replace(year=published.year + 1)
        except ValueError:
            fixed = None
        return (fixed if fixed and fixed >= earliest else earliest), True
    return published, False

pairs = inventory.apply(effective_date, axis=1)
inventory["published_effective"] = [p[0].isoformat() for p in pairs]
inventory["date_suspect"] = [p[1] for p in pairs]
print("suspect dates:", int(inventory.date_suspect.sum()))
display(inventory.loc[inventory.date_suspect, ["doc_id", "title", "published", "published_effective"]])

## Download

Sizes are checked with a HEAD request first. With `DRY_RUN = True` this cell only reports what would be fetched.

In [ ]:
ZIP_DIR = RAW_RAG / "_zips"

def remote_size(url):
    try:
        time.sleep(REQUEST_DELAY_S)
        r = SESSION.head(url, timeout=TIMEOUT, allow_redirects=True, verify=certifi.where())
        return int(r.headers["Content-Length"]) if "Content-Length" in r.headers else None
    except Exception:
        return None

def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def fetch_body(row):
    """Bytes of one document. Zip members come out of a zip that is downloaded once and cached."""
    if isinstance(row.zip_member, str):
        ZIP_DIR.mkdir(exist_ok=True)
        zpath = ZIP_DIR / (row.file_url.rstrip("/").split("/")[-2] + ".zip")
        if not zpath.is_file():
            zpath.write_bytes(get(row.file_url).content)
        with zipfile.ZipFile(zpath) as zf:
            return zf.read(row.zip_member)
    return get(row.file_url).content

todo = inventory.dropna(subset=["file_url"]).copy()
todo["path"] = todo.apply(lambda r: RAW_RAG / f"{r.doc_id}.{r.format}", axis=1)
todo["cached"] = todo["path"].map(lambda p: p.is_file())

if DRY_RUN:
    # Archive rows already know their size; the others get a HEAD request.
    todo["bytes"] = [p.stat().st_size if c else (e if pd.notna(e) else remote_size(u))
                     for u, c, p, e in zip(todo.file_url, todo.cached, todo.path, todo.expected_bytes)]
    print("DRY RUN: nothing downloaded")
    print("files to fetch:", int((~todo.cached).sum()), "| already cached:", int(todo.cached.sum()))
    print("approx total MB:", round(todo["bytes"].fillna(0).sum() / 1e6, 1),
          "| unknown sizes:", int(todo["bytes"].isna().sum()))
    display(todo.groupby("source_id").agg(files=("doc_id", "size"),
                                          MB=("bytes", lambda s: round(s.fillna(0).sum() / 1e6, 1))))
else:
    records = {}
    for row in todo.itertuples():
        if not row.cached:
            body = fetch_body(row)
            row.path.write_bytes(body)
        else:
            body = row.path.read_bytes()
        records[row.doc_id] = {"sha256": sha256_bytes(body), "bytes": len(body),
                               "path": str(row.path.relative_to(REPO))}
    print("downloaded/verified", len(records), "files")

## Text extraction

HTML pages use `trafilatura` (keeps tables); PDFs use `pypdf`. Extracted text is saved one file per document. Any document with very little text is flagged, since scanned PDFs need OCR and this notebook does not do that.

In [ ]:
MIN_CHARS = 500

def extract_text(path, fmt):
    if fmt == "html":
        import trafilatura
        return trafilatura.extract(path.read_text(encoding="utf-8", errors="ignore"),
                                   include_tables=True, include_comments=False) or ""
    from pypdf import PdfReader
    reader = PdfReader(str(path))
    return "\n\n".join((page.extract_text() or "") for page in reader.pages)

extraction = []
if DRY_RUN:
    print("DRY RUN: skipping extraction (no files downloaded)")
else:
    for row in todo.itertuples():
        text = extract_text(row.path, row.format)
        (TEXT_DIR / f"{row.doc_id}.txt").write_text(text, encoding="utf-8")
        extraction.append({"doc_id": row.doc_id, "chars": len(text), "low_text": len(text) < MIN_CHARS})
    report = pd.DataFrame(extraction)
    print("documents:", len(report), "| low text (<%d chars):" % MIN_CHARS, int(report.low_text.sum()))
    display(report.sort_values("chars").head(10))

## Provenance manifest

Follows the manifest convention of notebooks 01 and 05. Written only after a real download.

In [ ]:
if DRY_RUN:
    print("DRY RUN: manifest not written")
else:
    manifest = {
        "name": "JobAI RAG corpus v1",
        "created_utc": NOW_UTC.isoformat(),
        "sources": {"A": "Statistics Finland job vacancy releases", "B": "KEHA Employment Bulletins (DORIA)",
                    "B2": "TEM Employment Bulletins (government publication archive)"},
        "licence_notes": {
            "A": STAT_LICENCE,
            "B": "In Copyright (DORIA rights field, raw value kept per document as licence_raw); raw files are not committed.",
            "B2": "2013-2015 zips: CC BY-SA 4.0. 2016 onward: personal use only, commercial use prohibited. Raw files are not committed.",
        },
        "documents": [
            {**{k: (None if pd.isna(v) else v) for k, v in rec.items()},
             **records.get(rec["doc_id"], {}),
             "chars": next((e["chars"] for e in extraction if e["doc_id"] == rec["doc_id"]), None)}
            for rec in inventory.to_dict(orient="records")
        ],
        "discovery_errors": bulletin_errors,
        "packages": {n: importlib.metadata.version(n) for n in
                     ["requests", "trafilatura", "beautifulsoup4", "pypdf", "pandas"]},
    }
    out = MAN / f"rag_corpus_{TS}.json"
    out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))
    print("wrote:", out)

## Known gaps (as of 2026-09-21)

- **Source A** covers only the 3 newest releases. Older releases are listed by JavaScript on `stat.fi`; releases before April 2022 are in the Finna archive. Extending this needs a seed list of URLs or a headless browser.
- **Source B2** fills 2013 to early 2025, but **only in Finnish**, and its licence for 2016 onward is personal, non-commercial use. The 2013–2014 zip PDFs are small (about 0.1 MB) and may extract differently from newer layouts; check their text before trusting them.
- **Estimated dates:** zip members (2013–2015) have no publication date in the archive, so the last day of the following month is used (`date_source = estimated`).
- **Licences for B and B2 are restrictive** (In Copyright / personal use): raw files stay out of git. Confirm with KEHA before publishing the corpus or the index.
- **Sources C and D** (TEM forecast, Labour Force Barometer) are not in the `rag.yaml` whitelist and are not collected.
- **Scanned PDFs** are flagged as low-text but not OCR'd. None were found in the first run.
- **PDF text is noisy** (checked on the July 2026 bulletin). Map and chart labels come out as short fragments ("Uusimaa / 13,5"), words contain invisible soft hyphens (`Kanta\u00adHäme`), and numbers use decimal commas. Notebook 10 must strip soft hyphens and drop or merge label fragments before chunking. The useful narrative on vacancies is a short section of each bulletin, not the whole document.

**Next:** notebook 10 chunks and embeds this text and builds the index.